# Tinh dao ham tu dong (Automatic Differentiation) va Gradient Checking voi TensorFlow

**Muc tieu bai hoc:**
- Hieu **automatic differentiation (autodiff)** la gi, khac gi voi tinh dao ham bang tay (giai
  tich) va tinh dao ham xap xi (numerical/finite difference).
- Biet dung `tf.GradientTape` - cong cu cot loi cua TensorFlow de tinh gradient tu dong cho bat ky
  phep tinh khac nhau nao.
- Hieu ky thuat **gradient checking**: dung dao ham xap xi de KIEM CHUNG dao ham tinh boi autodiff/
  cong thuc giai tich - van huu ich ngay ca khi dung framework, moi khi ban tu viet mot phep tinh/
  loss moi chua co san.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/02_Debug-gradient-Gradient-Checking"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Ba cach tinh dao ham

Khi huan luyen mang no-ron bang gradient descent, ta can dao ham `dJ/d\theta` cua ham mat mat `J`
theo tung tham so `\theta`. Co 3 cach de co duoc con so nay:

1. **Dao ham giai tich (analytic/symbolic)** - tu tay ap dung quy tac chuoi (chain rule), suy ra
   cong thuc dong dang roi lap trinh dung cong thuc do. Nhanh va chinh xac **neu suy dung**, nhung
   de sai (nham dau, quen mot so hang, sai thu tu nhan ma tran...) khi ham phuc tap.
2. **Dao ham xap xi (numerical/finite difference)** - khong can biet cong thuc, chi can tinh
   `J` tai 2 diem gan `\theta`:
   $$\frac{dJ}{d\theta} \approx \frac{J(\theta+\varepsilon) - J(\theta-\varepsilon)}{2\varepsilon}$$
   Luon "dung xap xi" (voi `\varepsilon` du nho) nhung **cham** (phai goi lai `J` 2 lan cho MOI tham
   so rieng le) nen khong dung de huan luyen, chi dung de **kiem chung**.
3. **Dao ham tu dong (automatic differentiation - autodiff)** - each phep toan co ban (`+`, `*`,
   `exp`, matrix multiply...) deu co san "cong thuc dao ham cuc bo"; framework **tu dong ghi lai**
   moi phep toan da thuc hien (mot "computation graph"), roi ap dung chain rule tu dong nguoc lai
   tu output ve input. Ket qua **chinh xac tuyet doi** (khong xap xi, khong phai tu suy tay) va
   **nhanh** (mot lan lan truyen nguoc tinh duoc gradient cho MOI tham so cung luc) - day chinh la
   co che TensorFlow/PyTorch/Keras dung de huan luyen mang no-ron.

Autodiff **khong phai** la dao ham xap xi (no khong dung cong thuc sai phan huu han o buoc 2) -
day la nham lan pho bien. No cung khong doi hoi ban tu viet cong thuc dao ham (khac buoc 1) - ban
chi can dinh nghia phep tinh forward, framework tu suy ra dao ham.

## 2. `tf.GradientTape`

`tf.GradientTape` la cong cu autodiff cua TensorFlow: no "ghi bang" (record tape) moi phep toan
thuc hien tren cac `tf.Variable` ben trong khoi `with tf.GradientTape() as tape:`, sau do
`tape.gradient(y, x)` tra ve `dy/dx` bang cach phat lai (chain rule tu dong) doan bang da ghi.

### Bai tap - `dtheta_autodiff`

Vi du don gian nhat: `J(theta) = theta * x`, mot ham 1 bien. Ve mat giai tich, `dJ/dtheta = x`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay


def dtheta_autodiff(theta_value, x):
    '''
    Dung tf.GradientTape de tinh dJ/dtheta cho J(theta) = theta * x tai theta = theta_value.

    Returns: dtheta (python float)
    '''
    theta = tf.Variable(theta_value, dtype=tf.float64)
    # (~3 dong)
    # with tf.GradientTape() as tape:
    #     J = ...
    # dtheta = tape.gradient(J, theta)
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return float(dtheta)

In [ ]:
#@title Answer { display-mode: "form" }
def dtheta_autodiff(theta_value, x):
    '''
    Dung tf.GradientTape de tinh dJ/dtheta cho J(theta) = theta * x tai theta = theta_value.

    Returns: dtheta (python float)
    '''
    theta = tf.Variable(theta_value, dtype=tf.float64)
    # (~3 dong)
    # YOUR CODE STARTS HERE
    with tf.GradientTape() as tape:
        J = theta * x
    dtheta = tape.gradient(J, theta)
    # YOUR CODE ENDS HERE
    return float(dtheta)

In [ ]:
x, theta_value = 2.0, 4.0
dtheta = dtheta_autodiff(theta_value, x)
print(f"J(theta) = theta * x  tai theta={theta_value}, x={x}")
print(f"dtheta (tu GradientTape) = {dtheta}")
print(f"dtheta ly thuyet (= x)   = {x}")
assert abs(dtheta - x) < 1e-6, "GradientTape cho ket qua sai!"
print("\nDung - GradientTape tu suy ra dJ/dtheta = x ma khong can ai lap trinh cong thuc nay.")

## 3. Gradient checking cho mot mang no-ron

Autodiff luon dung **theo dinh nghia toan hoc** cua phep tinh ban da khai bao trong `GradientTape`.
Nhung neu ban tu dinh nghia mot phep tinh moi (mot custom loss, mot custom layer) va lo VIET SAI
phep tinh do, autodiff se tinh dao ham **chinh xac cho cong thuc sai** do - no khong tu phat hien
ra ban dinh dinh nghia sai cong thuc gi. Day la luc **gradient checking** (dung dao ham xap xi de
kiem chung) van con gia tri, du dang dung framework hien dai.

Ta thu ap dung ky thuat nay cho mot mang no-ron nho `[8, 5, 3, 1]`, tren mot mini-batch 16 diem
du lieu that lay tu bo du lieu **Pima Indians Diabetes** (768 benh nhan, 8 dac trung lam sang, du
doan tieu duong - se dung xuyen suot cac bai sau).

### Bai tap - `compute_gradients_autodiff`

In [ ]:
from keras_utils import load_dataset

train_X, train_Y, test_X, test_Y = load_dataset()

np.random.seed(1)
batch_idx = np.random.choice(train_X.shape[1], size=16, replace=False)
X_batch = train_X[:, batch_idx].T.astype("float32")  # (16, 8) - Keras quy uoc sample-first
Y_batch = train_Y[:, batch_idx].T.astype("float32")  # (16, 1)
print("X_batch:", X_batch.shape, " Y_batch:", Y_batch.shape)

In [ ]:
def compute_gradients_autodiff(model, X, Y):
    '''
    Dung tf.GradientTape de tinh gradient cua binary cross-entropy loss theo TOAN BO
    model.trainable_variables cua mot mang Keras.

    Arguments:
    model -- tf.keras.Model (Sequential) da co trong so
    X, Y  -- tensor du lieu vao / nhan, sample-first (shape (m, n_x) va (m, 1))

    Returns: (loss, grads) - grads la list, cung do dai/thu tu voi model.trainable_variables
    '''
    # (~4 dong)
    # with tf.GradientTape() as tape:
    #     Y_hat = ...
    #     loss = ...
    # grads = tape.gradient(loss, model.trainable_variables)
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return loss, grads

In [ ]:
#@title Answer { display-mode: "form" }
def compute_gradients_autodiff(model, X, Y):
    '''
    Dung tf.GradientTape de tinh gradient cua binary cross-entropy loss theo TOAN BO
    model.trainable_variables cua mot mang Keras.

    Arguments:
    model -- tf.keras.Model (Sequential) da co trong so
    X, Y  -- tensor du lieu vao / nhan, sample-first (shape (m, n_x) va (m, 1))

    Returns: (loss, grads) - grads la list, cung do dai/thu tu voi model.trainable_variables
    '''
    # (~4 dong)
    # YOUR CODE STARTS HERE
    with tf.GradientTape() as tape:
        Y_hat = model(X, training=False)
        loss = tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_hat))
    grads = tape.gradient(loss, model.trainable_variables)
    # YOUR CODE ENDS HERE
    return loss, grads

Ham `numerical_gradient_check` duoi day thuc hien dung dinh nghia toan hoc o muc 1 (cach 2): voi
moi toa do duoc chon ngau nhien trong `model.trainable_variables`, no tinh
`(J(theta+eps) - J(theta-eps)) / (2*eps)`, roi gop toan bo cac gia tri lai thanh **mot** sai so
tuong doi duy nhat bang cong thuc chuan:

$$\text{difference} = \frac{\|\text{grad}_{\text{autodiff}} - \text{grad}_{\text{approx}}\|}{\|\text{grad}_{\text{autodiff}}\| + \|\text{grad}_{\text{approx}}\|}$$

Gop bang norm (thay vi doc tung sai so rieng le) giup ket qua on dinh hon: voi mang dung ReLU, mot
vai toa do co the roi dung vao "diem gay" (kink) cua ReLU, noi dao ham mot ben (autodiff) va sai
phan hai ben (numerical) lech nhau du autodiff van hoan toan dung - day la han che da biet cua
numerical gradient checking tren ham khong muot hoan toan tai moi diem.

Nguong doc ket qua (quy uoc chuan trong kiem tra gradient, ap dung cho ham **muot** hoan toan):
`difference < 1e-7` la rat tot, `1e-7` - `1e-5` can xem xet them, `> 1e-3` gan nhu chac chan co loi
trong cong thuc. Vi mang o day dung **ReLU** (khong muot tai `0`, xem luu y ve "diem gay" phia
duoi), nguong thuc te se noi long hon mot chut: `difference` co the len toi ~`1e-2` ma van la
BINH THUONG (do vai toa do ngau nhien roi dung diem gay), trong khi mot loi cong thuc thuc su
thuong cho `difference` lon hon han (`> 0.1`, giong muc do sai thay o cac bai tap tu viet tay).

In [ ]:
def numerical_gradient_check(model, X, Y, grads, epsilon=1e-4, num_checks=30, seed=1):
    '''
    Kiem chung gradient (lay ngau nhien mot so toa do tu cac trainable_variables) bang sai phan
    huu han, gop lai thanh MOT sai so tuong doi duy nhat theo cong thuc norm o tren.
    '''
    rng = np.random.default_rng(seed)
    grad_approx_list, grad_autodiff_list = [], []
    for _ in range(num_checks):
        var_idx = int(rng.integers(0, len(model.trainable_variables)))
        var = model.trainable_variables[var_idx]
        flat_size = int(np.prod(var.shape))
        idx = np.unravel_index(int(rng.integers(0, flat_size)), var.shape)

        original = var.numpy()
        value0 = original[idx]

        perturbed = original.copy()
        perturbed[idx] = value0 + epsilon
        var.assign(perturbed)
        Y_plus = model(X, training=False)
        loss_plus = float(tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_plus)))

        perturbed[idx] = value0 - epsilon
        var.assign(perturbed)
        Y_minus = model(X, training=False)
        loss_minus = float(tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_minus)))

        var.assign(original)

        grad_approx_list.append((loss_plus - loss_minus) / (2 * epsilon))
        grad_autodiff_list.append(float(grads[var_idx].numpy()[idx]))

    grad_approx = np.array(grad_approx_list)
    grad_autodiff = np.array(grad_autodiff_list)
    difference = np.linalg.norm(grad_autodiff - grad_approx) / (
        np.linalg.norm(grad_autodiff) + np.linalg.norm(grad_approx)
    )
    return difference, grad_autodiff, grad_approx


tf.random.set_seed(4)
init = keras.initializers.GlorotUniform(seed=4)
model = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(5, activation="relu", kernel_initializer=init),
    layers.Dense(3, activation="relu", kernel_initializer=init),
    layers.Dense(1, activation="sigmoid", kernel_initializer=init),
])

loss, grads = compute_gradients_autodiff(model, tf.constant(X_batch), tf.constant(Y_batch))
print(f"Loss tren minibatch 16 diem: {loss:.4f}")

difference, grad_autodiff, grad_approx = numerical_gradient_check(
    model, tf.constant(X_batch), tf.constant(Y_batch), grads, num_checks=50,
)
print(f"\ndifference (50 toa do duoc lay mau, gop bang norm) = {difference:.2e}")
assert difference < 5e-3, "Gradient tu GradientTape khong khop voi numerical gradient!"
print("\nNho - GradientTape cho gradient CHINH XAC (autodiff, khong xap xi), gan nhu tuyet doi")
print("khop voi dao ham xap xi (numerical) tinh doc lap.")

## 4. Tong ket

- **Autodiff** (`tf.GradientTape`) tinh dao ham chinh xac bang chain rule tu dong, khong xap xi va
  khong doi hoi ban tu suy cong thuc - day la co che nen tang cho moi framework deep learning hien
  dai (TensorFlow, PyTorch, JAX...).
- **Numerical gradient** (sai phan huu han) van chinh xac ve mat toan hoc nhung qua cham de dung
  trong vong lap huan luyen - vai tro cua no la **cong cu kiem chung doc lap**, khong phai cong cu
  tinh gradient chinh.
- **Gradient checking** (so sanh 2 cach tren) van con gia tri thuc te ngay ca khi dung framework:
  no khong bao gio can thiet cho cac lop co san (`Dense`, `Conv2D`...) - nhung tro nen quan trong
  ngay khi ban **tu viet mot phep tinh/loss/layer moi** ma Keras chua co san (vi du: mot ham mat
  mat tuy chinh, hoac mot lop chuan hoa tu thiet ke) - luc do ban moi thuc su co nguy co dinh nghia
  sai cong thuc, va autodiff se "trung thanh" tinh dao ham dung cho ca cong thuc sai do.